# Tarefa 2 — Baseline por fluxo e rotulagem

Lê a tabela tratada da Tarefa 1 (`data/interim/medicoes.csv`, gerada a partir de `raw/`, que não é alterado) e produz:

1. a **ficha de baseline** de cada fluxo, calculada **só com o Período A**;
2. as métricas relativas e a **classe** (OK, RISCO, FALHA) de cada linha do **Período B**, pela tabela de regras da Tarefa 2;
3. o recorte temporal 50/20/30 dentro de B.

Nenhuma linha do Período A recebe classe nem entra no dataset. Parâmetros em `config/coleta_config.json`.

## 1. Configuração e leitura

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config" / "coleta_config.json").exists())
CFG = json.loads((RAIZ / "config" / "coleta_config.json").read_text(encoding="utf-8"))
DIR_INTERIM = RAIZ / CFG["caminhos"]["interim"]
DIR_PROC = RAIZ / CFG["caminhos"]["processed"]
DIR_PROC.mkdir(parents=True, exist_ok=True)

MIN_VALIDOS = CFG["rotulagem"]["min_rtt_validos"]
CORTE = CFG["rotulagem"]["corte"]

def epoch(iso):
    return int(pd.Timestamp(iso).timestamp())
A0, A1 = epoch(CFG["periodos"]["A"]["inicio"]), epoch(CFG["periodos"]["A"]["fim"])
B0, B1 = epoch(CFG["periodos"]["B"]["inicio"]), epoch(CFG["periodos"]["B"]["fim"])
assert A1 == B0, "A e B devem ser contíguos."

tab = pd.read_csv(DIR_INTERIM / "medicoes.csv")
tab["timestamp"] = pd.to_datetime(tab["timestamp"], utc=True)
seg = (tab["timestamp"] - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)
tab["periodo"] = np.select([(seg >= A0) & (seg < A1), (seg >= B0) & (seg < B1)], ["A", "B"], "fora")
print("Raiz:", RAIZ)

Raiz: c:\Users\duduf\icmpmaster


## 2. Inspeção antes de calcular

In [2]:
print("Linhas:", len(tab), "| fluxos:", tab["fluxo_id"].nunique())
print("Linhas por período:", tab["periodo"].value_counts().to_dict())
print("Duplicatas (fluxo_id, timestamp):", int(tab.duplicated(["fluxo_id", "timestamp"]).sum()))
print("RTT vazio:", int(tab["rtt_medio_ms"].isna().sum()), "| RTT = 0:", int((tab["rtt_medio_ms"] == 0).sum()))
assert (tab["rtt_medio_ms"] == 0).sum() == 0, "RTT vazio não pode ter virado 0."
fora = tab[tab["periodo"] == "fora"]
print("Linhas fora de A e B (não entram):", len(fora), "| fluxos só fora:",
      len(set(fora["fluxo_id"]) - set(tab.loc[tab["periodo"] != "fora", "fluxo_id"])))
uteis = tab[tab["periodo"] != "fora"]
print("Fluxos em A:", uteis[uteis.periodo == "A"]["fluxo_id"].nunique(),
      "| em B:", uteis[uteis.periodo == "B"]["fluxo_id"].nunique())

Linhas: 346569 | fluxos: 100
Linhas por período: {'A': 173676, 'B': 172094, 'fora': 799}
Duplicatas (fluxo_id, timestamp): 0
RTT vazio: 21640 | RTT = 0: 0
Linhas fora de A e B (não entram): 799 | fluxos só fora: 30
Fluxos em A: 70 | em B: 70


## 3. Ficha de baseline (Período A)

Por fluxo, só com linhas de A:

| Campo | Cálculo |
|---|---|
| `mediana` | mediana dos RTT válidos |
| `MAD` | `mediana(\|RTT − mediana\|)` nos RTT válidos |
| `iqr` | Q3 − Q1 dos RTT válidos (usado só quando `MAD = 0`) |
| `jitter_tipico` | mediana do `jitter_ms` |
| `perda_tipica` | mediana do `perda_pct` |
| `prop_resposta` | RTT válidos ÷ medições |

Fluxo com menos de **1.500 RTT válidos** em A fica `baseline_insuficiente`, sem mediana inventada, e não recebe classe.

In [3]:
def ficha(g):
    v = g["rtt_medio_ms"].dropna()
    n, nv = len(g), len(v)
    f = {"n_medicoes_A": n, "n_rtt_validos_A": nv, "mediana": np.nan, "MAD": np.nan, "iqr": np.nan,
         "jitter_tipico": np.nan, "perda_tipica": np.nan,
         "prop_resposta": nv / n if n else np.nan, "status": "baseline_insuficiente"}
    if nv >= MIN_VALIDOS:
        med = v.median()
        f.update(mediana=med, MAD=(v - med).abs().median(), iqr=v.quantile(.75) - v.quantile(.25),
                 jitter_tipico=g["jitter_ms"].median(), perda_tipica=g["perda_pct"].median(),
                 status="ok")
    return pd.Series(f)

A = tab[tab["periodo"] == "A"]
fluxos_ab = sorted(set(uteis["fluxo_id"]))
base = pd.DataFrame({k: ficha(g) for k, g in A.groupby("fluxo_id")}).T.reindex(fluxos_ab)
for c in base.columns.drop("status"):
    base[c] = pd.to_numeric(base[c])
base["status"] = base["status"].fillna("baseline_insuficiente")
base["n_medicoes_A"] = base["n_medicoes_A"].fillna(0).astype(int)
base["n_rtt_validos_A"] = base["n_rtt_validos_A"].fillna(0).astype(int)
base = base.reset_index().rename(columns={"index": "fluxo_id"})

print("Fluxos com baseline ok:", int((base.status == "ok").sum()),
      "| baseline_insuficiente:", int((base.status != "ok").sum()))
if (base.status != "ok").any():
    display(base.loc[base.status != "ok", ["fluxo_id", "n_medicoes_A", "n_rtt_validos_A", "status"]])
print("MAD = 0 em", int((base["MAD"] == 0).sum()), "fluxos | jitter_tipico = 0 em", int((base["jitter_tipico"] == 0).sum()))
base.to_csv(DIR_PROC / "baseline_por_fluxo.csv", index=False)
display(base[base.status == "ok"].head())

Fluxos com baseline ok: 65 | baseline_insuficiente: 5


,fluxo_id,n_medicoes_A,n_rtt_validos_A,status
20,11226|2001:503:c27::2:30,2520,0,baseline_insuficiente
35,52037|2001:503:c27::2:30,2520,0,baseline_insuficiente
37,52385|63.222.187.5,1327,1324,baseline_insuficiente
43,61240|193.1.33.11,2520,0,baseline_insuficiente
62,7212|148.163.220.5,1537,0,baseline_insuficiente


MAD = 0 em 0 fluxos | jitter_tipico = 0 em 0


,fluxo_id,n_medicoes_A,n_rtt_validos_A,mediana,MAD,iqr,jitter_tipico,perda_tipica,prop_resposta,status
0,1000755|63.222.187.5,2520,2517,254.832574,0.242517,0.545583,0.092127,0.0,0.998810,ok
1,1001602|193.1.33.11,2520,2520,38.646940,0.074550,0.158510,0.096695,0.0,1.000000,ok
2,1002564|63.222.187.5,2520,2516,206.193258,0.213829,0.432775,0.129016,0.0,0.998413,ok
3,1002754|63.222.187.5,2520,2518,42.184152,0.125583,0.332176,0.030004,0.0,0.999206,ok
4,1002841|63.222.187.5,2520,2518,57.835950,1.333060,2.569183,0.034560,0.0,0.999206,ok


### 3.1 Conferência: um caminho curto e um longo

As medianas devem ser diferentes. O tipo de caminho vem de `probes_paises.csv` (mesmo país = curto) e serve só para conferir; não é variável do modelo.

In [4]:
pp = DIR_INTERIM / "probes_paises.csv"
caminho = pd.read_csv(pp).drop_duplicates("fluxo_id").set_index("fluxo_id")["caminho"] if pp.exists() else pd.Series(dtype=str)
ok = base[base.status == "ok"].copy()
ok["caminho"] = ok["fluxo_id"].map(caminho)
for tipo in ("curto", "longo"):
    x = ok[ok["caminho"] == tipo]
    if len(x):
        r = x.sort_values("fluxo_id").iloc[0]
        print(f"{tipo:>5}: {r.fluxo_id} | mediana {r.mediana:.2f} ms | MAD {r.MAD:.2f} | n válidos {int(r.n_rtt_validos_A)}")
    else:
        print(f"{tipo:>5}: nenhum fluxo com baseline ok")

curto: 1010583|103.77.105.3 | mediana 7.21 ms | MAD 4.27 | n válidos 2520
longo: 1000755|63.222.187.5 | mediana 254.83 ms | MAD 0.24 | n válidos 2517


## 4. Métricas relativas do Período B

Só fluxos com baseline `ok`. Valem estas regras:

- `z_robusto = (RTT − mediana) / (1,4826 × MAD)`. Se `MAD = 0`, o denominador é `max(IQR/1,349; 1 ms)`. Se `RTT = mediana`, `z_robusto = 0`.
- `aumento_pct = (RTT − mediana) / mediana × 100`.
- `jitter_relativo = jitter_atual / jitter_tipico`. Se `jitter_tipico = 0`: jitter atual 0 vale 1; jitter atual > 0 vale 3 (cai no critério `≥ 3`). Sem jitter atual, fica vazio.
- Linha sem RTT (timeout): `z_robusto`, `aumento_pct` e `jitter_relativo` ficam vazios. O timeout é tratado por `timeout_atual`.
- `n5_*` olham as últimas 5 medições **do mesmo fluxo dentro de B** (a atual e as 4 anteriores; no começo de B a janela é menor). B não depende de A além da ficha.

In [5]:
ok_fichas = base[base.status == "ok"][["fluxo_id", "mediana", "MAD", "iqr", "jitter_tipico"]]
b = tab[tab["periodo"] == "B"].merge(ok_fichas, on="fluxo_id", how="inner")
b = b.sort_values(["fluxo_id", "timestamp"]).reset_index(drop=True)

rtt, dif = b["rtt_medio_ms"], b["rtt_medio_ms"] - b["mediana"]
den = np.where(b["MAD"] > 0, 1.4826 * b["MAD"], np.maximum(b["iqr"] / 1.349, 1.0))
b["z_robusto"] = np.where(rtt.isna(), np.nan, np.where(dif == 0, 0.0, dif / den))
b["aumento_pct"] = np.where(rtt.isna(), np.nan, dif / b["mediana"] * 100)

jt, j = b["jitter_tipico"], b["jitter_ms"]
b["jitter_relativo"] = np.select(
    [(j.isna() | jt.isna()).to_numpy(), (jt == 0).to_numpy()],
    [np.nan, np.where(j == 0, 1.0, 3.0)],
    default=(j / jt.where(jt > 0)).to_numpy())

b["timeout_atual"] = (rtt.isna() | (b["perda_pct"] == 100)).astype(int)

crit5 = (((b["z_robusto"] >= 2) & (b["z_robusto"] < 3.5))
         | ((b["aumento_pct"] >= 30) & (b["aumento_pct"] <= 80))
         | (b["jitter_relativo"] >= 3))
ind = pd.DataFrame({"n5_timeout": b["timeout_atual"],
                    "n5_aumento80": (b["aumento_pct"] > 80).astype(int),
                    "n5_risco": crit5.astype(int)})
for c in ind:
    b[c] = ind[c].groupby(b["fluxo_id"]).transform(lambda s: s.rolling(5, min_periods=1).sum()).astype(int)
b["crit5"] = crit5
print("Linhas de B com baseline ok:", len(b), "| fluxos:", b["fluxo_id"].nunique())
display(b[["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct"]].describe().T)

Linhas de B com baseline ok: 160885 | fluxos: 65


,count,mean,std,min,25%,50%,75%,max
z_robusto,160061.0,3.069388,43.003935,-165.773638,-0.741077,0.049362,1.467510,2818.765599
aumento_pct,160061.0,8.607323,69.837677,-82.539242,-0.691924,0.022895,1.597749,2375.921642
jitter_relativo,159395.0,2.848318,41.021171,0.000077,0.610620,1.004044,1.611257,9432.822330
perda_pct,160885.0,0.754265,7.904448,0.000000,0.000000,0.000000,0.000000,100.000000


## 5. Rotulagem

Ordem fixa; a primeira condição verdadeira decide (FALHA > RISCO > OK):

| Ordem | Condição | Classe |
|---|---|---|
| 1 | `perda_pct ≥ 10` | FALHA |
| 2 | `n5_timeout ≥ 3` | FALHA |
| 3 | RTT presente e `z_robusto ≥ 3,5` | FALHA |
| 4 | `n5_aumento80 ≥ 2` | FALHA |
| 5 | (`2 ≤ z_robusto < 3,5` ou `30 ≤ aumento_pct ≤ 80` ou `jitter_relativo ≥ 3`) e `n5_risco ≥ 2` | RISCO |
| 6 | nenhuma acima | OK |

In [6]:
r1 = (b["perda_pct"] >= 10).to_numpy()
r2 = (b["n5_timeout"] >= 3).to_numpy()
r3 = (b["z_robusto"] >= 3.5).to_numpy()
r4 = (b["n5_aumento80"] >= 2).to_numpy()
r5 = (b["crit5"] & (b["n5_risco"] >= 2)).to_numpy()
b["ordem"] = np.select([r1, r2, r3, r4, r5], [1, 2, 3, 4, 5], 6)
b["classe"] = np.select([b["ordem"] <= 4, b["ordem"] == 5], ["FALHA", "RISCO"], "OK")

cont = b["classe"].value_counts()
display(pd.DataFrame({"linhas": cont, "%": (cont / len(b) * 100).round(2)}))
print("Ordem que decidiu a classe:", b["ordem"].value_counts().sort_index().to_dict())

,linhas,%
classe,,
OK,125258,77.86
FALHA,25006,15.54
RISCO,10621,6.60


Ordem que decidiu a classe: {1: 1693, 2: 98, 3: 21533, 4: 1682, 5: 10621, 6: 125258}


### 5.1 Três exemplos reais (com as métricas e a ordem que disparou)

In [7]:
cols = ["fluxo_id", "timestamp", "rtt_medio_ms", "mediana", "MAD", "z_robusto", "aumento_pct",
        "jitter_relativo", "perda_pct", "n5_timeout", "n5_aumento80", "n5_risco", "ordem", "classe"]
b["caminho"] = b["fluxo_id"].map(caminho)
exemplos = []
ok_ = b[(b.classe == "OK") & (b.z_robusto.abs() < 1)].sort_values("mediana", ascending=False)
if len(ok_): exemplos.append(("OK em caminho de RTT alto (RTT alto não é sinal de problema)", ok_.iloc[0]))
ri = b[b.classe == "RISCO"]
if len(ri): exemplos.append(("RISCO", ri.iloc[len(ri) // 2]))
fa = b[b.classe == "FALHA"]
fa_c = fa[fa.caminho == "curto"]
if len(fa): exemplos.append(("FALHA", (fa_c if len(fa_c) else fa).iloc[0]))
for titulo, r in exemplos:
    print("=" * 70); print(titulo)
    print(r[cols].to_string())

OK em caminho de RTT alto (RTT alto não é sinal de problema)
fluxo_id                  7596|148.163.220.5
timestamp          2026-09-13 13:53:10+00:00
rtt_medio_ms                      375.594663
mediana                           375.531461
MAD                                 0.171047
z_robusto                           0.249222
aumento_pct                          0.01683
jitter_relativo                     0.999135
perda_pct                                0.0
n5_timeout                                 0
n5_aumento80                               0
n5_risco                                   0
ordem                                      6
classe                                    OK
RISCO
fluxo_id                   6380|87.238.48.84
timestamp          2026-09-19 12:21:23+00:00
rtt_medio_ms                      126.518335
mediana                           126.240752
MAD                                 0.071907
z_robusto                            2.60376
aumento_pct                      

## 6. Recorte temporal dentro de B

Em cada fluxo, por ordem de tempo: primeiros 50% = treino, 20% seguintes = validação, últimos 30% = teste (valores no config). Não há embaralhamento e nenhuma linha do Período A entra.

In [8]:
pos = b.groupby("fluxo_id").cumcount()
n = b.groupby("fluxo_id")["fluxo_id"].transform("size")
frac = pos / n
b["particao"] = np.select([frac < CORTE["treino"], frac < CORTE["treino"] + CORTE["validacao"]],
                          ["treino", "validacao"], "teste")

resumo = b.groupby("particao").agg(linhas=("classe", "size"), inicio=("timestamp", "min"), fim=("timestamp", "max"))
resumo = resumo.join(pd.crosstab(b["particao"], b["classe"])).reindex(["treino", "validacao", "teste"])
display(resumo)
for blocos, g in b.groupby("particao"):
    if "RISCO" not in set(g["classe"]):
        print(f"ATENÇÃO: o bloco {blocos} ficou sem RISCO; as proporções precisam ser reavaliadas.")
assert b["timestamp"].min() >= pd.Timestamp(B0, unit="s", tz="UTC"), "Linha de A no dataset."

,linhas,inicio,fim,FALHA,OK,RISCO
particao,,,,,,
treino,80450,2026-09-13 04:32:18+00:00,2026-09-16 18:34:56+00:00,12810,62788,4852
validacao,32182,2026-09-13 15:14:42+00:00,2026-09-18 02:37:50+00:00,4946,24738,2498
teste,48253,2026-09-13 19:26:51+00:00,2026-09-20 04:31:53+00:00,7250,37732,3271


## 7. Gravar o dataset rotulado do Período B

In [9]:
SAIDA = ["fluxo_id", "timestamp", "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
         "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco", "classe", "particao"]
dataset = b[SAIDA]
dataset.to_csv(DIR_PROC / "dataset_rotulado_B.csv", index=False)
print("Gravado:", DIR_PROC / "dataset_rotulado_B.csv", dataset.shape)
display(dataset.head())

Gravado: c:\Users\duduf\icmpmaster\data\processed\dataset_rotulado_B.csv (160885, 12)


,fluxo_id,timestamp,z_robusto,aumento_pct,jitter_relativo,perda_pct,timeout_atual,n5_timeout,n5_aumento80,n5_risco,classe,particao
0,1000755|63.222.187.5,2026-09-13 04:35:00+00:00,0.951096,0.134195,1.095878,0.0,0,0,0,0,OK,treino
1,1000755|63.222.187.5,2026-09-13 04:39:04+00:00,-0.556732,-0.078552,1.046818,0.0,0,0,0,0,OK,treino
2,1000755|63.222.187.5,2026-09-13 04:42:58+00:00,0.679571,0.095884,1.785438,0.0,0,0,0,0,OK,treino
3,1000755|63.222.187.5,2026-09-13 04:47:03+00:00,0.574938,0.081121,1.057132,0.0,0,0,0,0,OK,treino
4,1000755|63.222.187.5,2026-09-13 04:51:03+00:00,1.741387,0.245701,0.768646,0.0,0,0,0,0,OK,treino


## 8. Números para o diário

In [10]:
print("N de registros usados (linhas de A e B nos fluxos):", len(uteis))
print("Fluxos totais:", base["fluxo_id"].nunique(), "| baseline ok:", int((base.status == "ok").sum()),
      "| baseline_insuficiente:", int((base.status != "ok").sum()))
print("Linhas do Período A:", int((tab.periodo == "A").sum()), "| Período B (com baseline ok):", len(b))
print("Classes:", b["classe"].value_counts().to_dict())
print("Blocos:", b["particao"].value_counts().reindex(["treino", "validacao", "teste"]).to_dict())
for k, g in b.groupby("particao"):
    print(f"  {k}: {g['timestamp'].min():%Y-%m-%d %H:%M} a {g['timestamp'].max():%Y-%m-%d %H:%M}")

N de registros usados (linhas de A e B nos fluxos): 345770
Fluxos totais: 70 | baseline ok: 65 | baseline_insuficiente: 5
Linhas do Período A: 173676 | Período B (com baseline ok): 160885
Classes: {'OK': 125258, 'FALHA': 25006, 'RISCO': 10621}
Blocos: {'treino': 80450, 'validacao': 32182, 'teste': 48253}
  teste: 2026-09-13 19:26 a 2026-09-20 04:31
  treino: 2026-09-13 04:32 a 2026-09-16 18:34
  validacao: 2026-09-13 15:14 a 2026-09-18 02:37
